In [ ]:
!https://github.com/d4-5/NLP3.git
%cd NLP3

In [ ]:
!pip install -r requirements.txt

In [ ]:
import stanza
from pathlib import Path
from src.ling_features import build_pipeline, extract_ling_features

stanza.download("uk", processors="tokenize,pos,lemma")

nlp = build_pipeline()

2026-03-02 19:05:34 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/resources.json
2026-03-02 19:05:34 WARNING: Language uk package default expects mwt, which has been added
2026-03-02 19:05:34 INFO: Downloading these customized packages for language: uk (Ukrainian)...
| Processor       | Package     |
---------------------------------
| tokenize        | iu          |
| mwt             | iu          |
| pos             | iu_charlm   |
| lemma           | iu_nocharlm |
| backward_charlm | conll17     |
| pretrain        | conll17     |
| forward_charlm  | conll17     |

2026-03-02 19:05:35 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/uk/tokenize/iu.pt
2026-03-02 19:05:35 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/uk/mwt/iu.pt
2026-03-02 19:05:43 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/uk/pos/iu_charlm.pt
2026-03-02 19:05:44 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/uk/le

In [ ]:
import pandas as pd

df = pd.read_csv("data/processed_v2.csv")
df.head()

,text_id,text_clean,sentences
0,text_0,"Дорогою зупинялися в селах, старцювали. Устимк...","['Дорогою зупинялися в селах, старцювали.', 'У..."
1,text_1,"То молитви продавав свої спудейські, то щось к...","['То молитви продавав свої спудейські, то щось..."
2,text_2,"Дуже часто дід досить точно вказував, куди сам...","['Дуже часто дід досить точно вказував, куди с..."
3,text_3,"Часом Амфібрахій, керуючись настановами старог...","['Часом Амфібрахій, керуючись настановами стар..."
4,text_4,Чи вірив у все це сам Амфібрахій?,['Чи вірив у все це сам Амфібрахій?']


In [11]:
sample_text = df.iloc[0]["text_clean"]

features = extract_ling_features(sample_text, nlp)

print("LEMMA TEXT:")
print(features["lemma_text"])

print("\nPOS TEXT:")
print(features["pos_text"])

LEMMA TEXT:
дорога зупинятися в село , старцювати . Устимко приспівувати дід . амфібрахій шукати підробітка .

POS TEXT:
NOUN VERB ADP NOUN PUNCT VERB PUNCT PROPN VERB NOUN PUNCT NOUN VERB NOUN PUNCT


In [17]:
def extract_names_pos(doc):
    names = []
    for sent in doc.sentences:
        words = sent.words
        current_name = []
        for word in words:
            if word.upos == "PROPN":
                current_name.append(word.text)
            else:
                if len(current_name) >= 2:
                    names.append(" ".join(current_name))
                current_name = []
        if len(current_name) >= 2:
            names.append(" ".join(current_name))
    return names
import re

def extract_names_regex(text):
    pattern = r"[А-ЯІЇЄҐ][а-яіїєґ']+(?:[-'][А-ЯІЇЄҐ][а-яіїєґ']+)?(?:\s[А-ЯІЇЄҐ][а-яіїєґ']+(?:[-'][А-ЯІЇЄҐ][а-яіїєґ']+)?)+"
    return re.findall(pattern, text)

In [ ]:
import json

edge_cases_path = Path("tests/ling_edge_cases.jsonl")
edge_cases = []
with edge_cases_path.open(encoding="utf-8") as f:
    for line in f:
        edge_cases.append(json.loads(line))

results = []
for case in edge_cases:
    raw_text = case["raw_text"]
    doc = nlp(raw_text)
    
    baseline1 = extract_names_regex(raw_text)
    baseline2 = extract_names_pos(doc)
    
    results.append({
        "raw": raw_text,
        "baseline1": baseline1,
        "baseline2": baseline2,
    })

results_df = pd.DataFrame(results)

pd.set_option("max_colwidth", 100)
results_df[["raw", "baseline1", "baseline2"]]

,raw,baseline1,baseline2
0,"Дід би все одно ніколи не дізнався, говорив він із кимось чи ні, але дурити сліпого... не по-люд...",[],[]
1,"Разом їли куліш і кашу, говорили про життя, про джуму, про козаків, про батька Сагайдачного і До...",[],[]
2,"Передасте?""",[],[]
3,"Що це - наївність чи безмежна довіра до тих, хто несе слово правди? ""А може, щось йому ще перека...",[],[]
4,"Пакуночки вони передавали за призначенням, а листи Амфібрахій зачитував адресатам. Устимко завжд...",[],[]
5,"Я хотів би прочитати щось твоє, може, ""Світло і сповідь""?",[],[]
6,167.2 - 167.6 ст.,[],[]
7,"- 5 та 0 % бази оподаткування у випадках, прямо визначених розділом IV ПКУ, зокрема при операція...",[],[]
8,"Усі зауваження й побажання просимо надсилати на адресою: Київ-1, вул. Грушевського, 4, Інститут ...",[],[НАН України]
9,"Як це близько до ""Люби ближнього свого""!",[],[]


Baseline 1 (regex): Precision = 1.0
Baseline 2 (POS/леми): Precision ≈ 0.8

У нашому NER-завданні ми помітили, що леми та POS не завжди допомагають витягати сутності. Нестандартні форми, скорочення та трансліт, наприклад слова на латиниці, іноді призводять до того, що POS ставить невірний тег. Власні назви та бренди з дефісами або апострофами, а також багатослівні імена чи організації, можуть пропускатися POS-базовим методом. Рідкісні словосполучення або цитати також іноді призводять до неправильного формування послідовності PROPN. Леми та POS не додали користі у точності для цього завдання і іноді пропускали сутності.